# 09 — Enrich Part D with by-Provider features (AE dataset)

Same method as Notebook 03 (Part B) and the paper:
- Inner join of the labeled aggregated Part D file with Part D **by Provider** on NPI + year.
- Missing values imputed with 0. Race and suppression flags are not used (already absent from Rahi's cleaned files).
- Location and descriptor text columns are dropped (city, state, FIPS, ZIP, RUCA, prescriber type, type source, entity code). Prescriber type is already in the aggregated file.

Input:
- `processed/part_d/labeled/part_d_A_labeled_2019_2023.parquet` (Notebook 07)
- `raw/part_d/sbp_provider_{2019..2023}.parquet` (Rahi; 60 columns, identical across years)

Output: `processed/part_d/labeled/part_d_AE_labeled_2019_2023.parquet` — labels from Notebook 07 carried over unchanged.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
RAW_D    = DATA_DIR / "raw" / "part_d"
LAB_DIR  = DATA_DIR / "processed" / "part_d" / "labeled"
A_LAB    = LAB_DIR / "part_d_A_labeled_2019_2023.parquet"
OUT      = LAB_DIR / "part_d_AE_labeled_2019_2023.parquet"
YEARS    = [2019, 2020, 2021, 2022, 2023]

DROP_TEXT = ["prscrbr_ent_cd", "prscrbr_city", "prscrbr_state_abrvtn", "prscrbr_state_fips",
             "prscrbr_zip5", "prscrbr_ruca", "prscrbr_type", "prscrbr_type_src"]
print(A_LAB.exists(), [ (RAW_D / f"sbp_provider_{y}.parquet").exists() for y in YEARS ])

True [True, True, True, True, True]


## Step 1 — Load by-Provider files and check them

In [7]:
frames = []
for y in YEARS:
    d = pd.read_parquet(RAW_D / f"sbp_provider_{y}.parquet")
    assert (d.year == y).all(), f"{y}: year column does not match file"
    assert not d.prscrbr_npi.duplicated().any(), f"{y}: duplicate NPI"
    frames.append(d)
sbp = pd.concat(frames, ignore_index=True); del frames
sbp["year"] = sbp["year"].astype("int64")
print(sbp.shape)
print("rows per year:\n", sbp.groupby("year").size())

FEATS = [c for c in sbp.columns if c not in DROP_TEXT + ["prscrbr_npi", "year"]]
non_num = [c for c in FEATS if not pd.api.types.is_numeric_dtype(sbp[c])]
assert not non_num, non_num
print("by-Provider features:", len(FEATS))

(6496198, 60)
rows per year:
 year
2019    1240595
2020    1255175
2021    1287454
2022    1332309
2023    1380665
dtype: int64
by-Provider features: 50


In [3]:
# Missingness by year (before 0-imputation). Heavily suppressed columns,
# and columns whose missingness changes by year, can create artificial drift.
miss = sbp.groupby("year")[FEATS].apply(lambda g: g.isna().mean()).T
miss = miss[miss.max(axis=1) > 0].round(3)
print(f"{len(miss)} of {len(FEATS)} features have missing values")
print(miss.sort_values(2023, ascending=False).to_string())

0 of 50 features have missing values
Empty DataFrame
Columns: [2019, 2020, 2021, 2022, 2023]
Index: []


## Step 2 — Join to the labeled aggregated file

In [4]:
a = pd.read_parquet(A_LAB)
n0 = len(a)
ae = a.merge(sbp.rename(columns={"prscrbr_npi": "npi"}), on=["npi", "year"], how="inner")

lost = a.groupby("year").size() - ae.groupby("year").size()
print("aggregated rows lost in join, per year:\n", lost.fillna(0).astype(int))
print(f"total kept {len(ae):,} of {n0:,}")
label_cols = [c for c in a.columns if c == "fraud" or c.startswith("fraud_h")]
print("fraud rows before/after:", {c: (int(a[c].sum()), int(ae[c].sum())) for c in label_cols})
del a

aggregated rows lost in join, per year:
 year
2019    0
2020    0
2021    0
2022    0
2023    0
dtype: int64
total kept 5,148,303 of 5,148,303
fraud rows before/after: {'fraud': (3274, 3274), 'fraud_h2': (958, 958), 'fraud_h3': (1653, 1653)}


In [5]:
# Sanity: the two files should describe the same provider.
same_type = (ae.provider_type.str.strip().str.lower() == ae.prscrbr_type.str.strip().str.lower()).mean()
print(f"provider type agrees: {same_type:.2%}")
# Drug-level rows are a subset of the provider total (CMS suppresses drugs with < 11 claims),
# so total_claims_sum should never exceed tot_clms.
over = (ae.total_claims_sum > ae.tot_clms + 0.5).mean()
print(f"rows where drug-level claims exceed provider total: {over:.4%}")

provider type agrees: 87.31%
rows where drug-level claims exceed provider total: 0.0000%


In [6]:
ae = ae.drop(columns=DROP_TEXT)
ae[FEATS] = ae[FEATS].fillna(0).astype("float32")      # paper: impute 0
assert ae.notna().all().all()
assert not ae.duplicated(["npi", "year"]).any()
print(ae.shape)
ae.to_parquet(OUT, index=False)
print(OUT)

(5148303, 86)
/scratch/sm3664/medicare-fraud/processed/part_d/labeled/part_d_AE_labeled_2019_2023.parquet
